 ## C4CSS Project: 
 
 ### a replication study for lexicon-based sentiment analysis

Use this baseline notebook to start preparing your C4CSS Exam project: a replication study of 


Chan et al. (2023) [A Comparative Study of Lexicon-Based Sentiment Analysis Methods](https://journal.computationalcommunication.org/article/view/4731).





To simplify things, the lexica have been downloaded and stored in the `downloaded_lexica` folder.

Some baseline code to load the lexica and apply them to sample input phrases is provided below: start from here to develop your project.

In [ ]:
# locate the lexica on your computer

DATA = './downloaded_lexica/'

AFINN = 'AFINN-111.txt'

NRC = 'NRC-Emotion-Lexicon-Wordlevel-v0.92.txt'

BING_POS = 'bing-positive-words.txt'

BING_NEG = 'bing-negative-words.txt'


In [ ]:
import pandas as pd
import os
from collections import Counter


####  Load Lexicon data

I had to write a different function for each lexicon; each returns a Pandas DataFrame with columns: 'word' and 'sentiment_score".

TODO: refactor with the lexicon as argument to a single function, or introduce a decorator.


In [ ]:

def load_afinn_lexicon(filepath=DATA+AFINN):
    '''
    Load AFINN lexicon from text file
    Expected format: word\tscore (tab-separated)
    
    Example:
        abandon	-2
        abandoned	-2
    '''

    mydata = {'word': [], 'score': []}
    
    with open(filepath, 'r', encoding='utf-8') as f:

        for line in f:
            # assume lines are 'well behaved'
            line = line.strip()
            
            if line:
                parts = line.split('\t')

                if len(parts) == 2:
                    mydata['word'].append(parts[0])
                    mydata['score'].append(int(parts[1]))
                else:
                    print(f'Possible lexicon corruption: {line}')
    
    return pd.DataFrame(mydata)

In [ ]:

def load_bing_lexicon(filepath=DATA):
    '''
    Load the Bing lexica from text files (tsv format)
    This assumes that BING_NEG also exists
    
    Example positive:
        a+
        abound
        abounds
    Example negative:
        2-faced
        2-faces
        abnormal
    '''

    mydata = {'word': [], 'sentiment': []}
    
    with open(filepath+BING_POS, 'r', encoding='utf-8') as f:

        for line in f:
            line = line.strip()

            if line:
                parts = line.split('\t')

                # unlikely
                if len(parts) == 2:
                    mydata['word'].append(parts[0])
                    mydata['sentiment'].append(parts[1])

                else:
                    # lukely, this is the 'positive' case
                    mydata['word'].append(parts[0])
                    mydata['sentiment'].append('positive')

    with open(filepath+BING_NEG, 'r', encoding='utf-8') as f:

        for line in f:
            line = line.strip()

            if line:
                parts = line.split('\t')

                # unlikely
                if len(parts) == 2:
                    mydata['word'].append(parts[0])
                    mydata['sentiment'].append(parts[1])

                else:
                    # lukely, this is the 'negative' case
                    mydata['word'].append(parts[0])
                    mydata['sentiment'].append('negative')
    
    return pd.DataFrame(mydata)

In [ ]:

def load_nrc_lexicon(filepath=DATA+NRC):
    '''
    Load the NRC lexicon from TSV text file
    Expected format: word\tsentiment (tab-separated)
    Note: Words can appear multiple times with different emotions
    
    Example:
        love	anger	0
        love	anticipation	0
        love	disgust	0
        love	fear	0
        love	joy	1
        love	negative	0
        love	positive	1
        love	sadness	0
        love	surprise	0
        love	trust	0
    '''

    data = {'word': [], 'sentiment': [], 'score': []}
    
    with open(filepath, 'r', encoding='utf-8') as f:

        for line in f:
            line = line.strip()
        
            if line:
                parts = line.split('\t')
        
                if len(parts) == 3:
                    data['word'].append(parts[0])
                    data['sentiment'].append(parts[1])
                    data['score'].append(int(parts[2]))
    
    return pd.DataFrame(data)

In [ ]:
def load_lexicon(lexicon_name=AFINN):
    '''
    Load 'afinn', 'bing', or 'nrc' and return a dataframe with word: value pairs
    
    Returns:
        pd.DataFrame: The loaded lexicon
    '''

    # I define a dictionary of loaders
    lexicon_loaders = {
        'afinn': load_afinn_lexicon,
        'bing': load_bing_lexicon,
        'nrc': load_nrc_lexicon
    }
    
    if lexicon_name.lower() not in lexicon_loaders:
        raise ValueError(f"Unknown lexicon: {lexicon_name}. Choose from: {list(lexicon_loaders.keys())}")
    
    return lexicon_loaders[lexicon_name.lower()]()


#### Sentiment analysis

TODO: refactor with the lexicon as argument to a single function, or introduce a decorator.

In [ ]:
def analyse_afinn(phrase, lexicon):
    '''
    Analyze sentiment using AFINN lexicon
    Returns total score and matched words
    '''

    # from phrase to a bag of words, all-lowercase and split at spaces ' '
    words = phrase.lower().split()
    
    afinn_dict = dict(zip(lexicon['word'], lexicon['score']))

    # for each word in the phrase, I seek it in the lexicon and, if it exists, add a pair to word_scores
    # TODO: unroll it for educational purposes
    word_scores = [(word, afinn_dict[word]) for word in words if word in afinn_dict]

    total_score = sum(score for _, score in word_scores)
    
    return {
        'lexicon': 'AFINN',
        'total_score': total_score,
        'matched_words': len(word_scores),
        'word_scores': word_scores,
        'average_score': total_score / len(word_scores) if word_scores else 0
    }

In [ ]:
def analyse_bing(phrase, lexicon):
    '''
    Analyze sentiment using Bing lexicon
    Returns positive/negative word counts
    '''

    # from phrase to a bag of words, all-lowercase and split at spaces ' '
    words = phrase.lower().split()

    positive_words = set(lexicon[lexicon['sentiment'] == 'positive']['word'])
    
    negative_words = set(lexicon[lexicon['sentiment'] == 'negative']['word'])
    
    pos_matches = [w for w in words if w in positive_words]

    neg_matches = [w for w in words if w in negative_words]

    return {
        'lexicon': 'Bing (pos and neg)',
        'positive_count': len(pos_matches),
        'negative_count': len(neg_matches),
        'net_sentiment': len(pos_matches) - len(neg_matches),
        'positive_words': pos_matches,
        'negative_words': neg_matches
    }

In [ ]:
def analyse_nrc(phrase, lexicon):
    '''
    Analyze emotions using NRC lexicon
    
    NB this is the trickiest case as we need to collect emotional dimensions

    Returns emotion counts and matched words
    '''

    # from phrase to a bag of words, all-lowercase and split at spaces ' '
    words = phrase.lower().split()

    # slightly involved implementation of counting how many emotions each word raises
    emotion_counts = Counter()
    
    matched_words = {}
    
    for word in words:
        
        # TODO: only sentiments that score 1 should be considered
        # Example:
        # abandoned	anger	1
        # abandoned	anticipation	0
        # abandoned	disgust	0
        # abandoned	fear	1
        # abandoned	joy	0
        # abandoned	negative	1
        # abandoned	positive	0
        # abandoned	sadness	1
        # abandoned	surprise	0
        # abandoned	trust	0
        # only scores for 'anger,' 'fear,' 'negative' and 'sadness'
        # 
        word_emotions = lexicon[lexicon['word'] == word]['sentiment'].values

        for emotion in word_emotions:

            emotion_counts[emotion] += 1

            if emotion not in matched_words:
                matched_words[emotion] = []
        
            matched_words[emotion].append(word)
    
    return {
        'lexicon': 'NRC',
        'emotion_counts': dict(emotion_counts),
        'matched_words': matched_words,
        'total_matches': sum(emotion_counts.values())
    }

In [ ]:

def score_phrase(phrase, selector):
    '''
    score the sentiment in phrase using one lexicon (default AFINN)
    
    it returns the score as a dictionary
    '''

    mydf = load_lexicon(selector)

    if selector == 'afinn':
        return analyse_afinn(phrase, mydf)
    
    elif selector == 'bing':
        return analyse_bing(phrase, mydf)

    elif selector == 'nrc':
        return analyse_nrc(phrase, mydf)
    
    else:
        print('unknown lexicon!')
        return None


#### Example call (1-shot)

In [ ]:
# Example phrase: afinn and bing are unable to pick up a signal
myphrase = "I love you but I hate the current political situation"


In [ ]:

result = score_phrase(myphrase, 'afinn')

print(f"\n{result['lexicon']} Analysis for '{myphrase}'")

# the scoring function returns a dictionary, I print it but can't generalise, see below
print(f"  Total Score: {result['total_score']}")
print(f"  Average Score: {result['average_score']}")
print(f"  Matched Words: {result['matched_words']}")
print(f"  Word Scores: {result['word_scores']}")

In [ ]:

result = score_phrase(myphrase, 'bing')

print(f"\n{result['lexicon']} Analysis for '{myphrase}'")

print(f"  Positive Count: {result['positive_count']}")
print(f"  Negative Count: {result['negative_count']}")
print(f"  Net Sentiment: {result['net_sentiment']}")
print(f"  Positive Words: {result['positive_words']}")
print(f"  Negative Words: {result['negative_words']}")


In [ ]:
result = score_phrase(myphrase, 'nrc')

print(f"\n{result['lexicon']} Analysis for '{myphrase}'")

print(f"  Total Matches: {result['total_matches']}")
print(f"  Matched words: {result['matched_words']}")

# mind the nested dictionary
print(f"  Emotion Counts: {result['emotion_counts']}")


In [ ]:
# TODO for students: go from 'parallel' counting the positives and negatives to an actual score

In [ ]:

print(f"\n  Matched Words by emotion:")

for emotion, words in result['matched_words'].items():
    print(f"    {emotion}: {words}")

#### To do:

- visualisation
- batch elaboration: we should be able to process a long text phrase by phrase calling the functions above
  

#### Batch Analysis: analyze multiple phrases at once